# But what is $\dfrac{dy}{dx}$ ?

From the slope of a line all the way to backpropagation, one idea at a time:

1. Slope of a straight line
2. $f(x) = x^2$: what happens when the two points get closer and closer (limits)
3. Velocity: change of distance over time
4. $f(x) = \dfrac{1}{x}$
5. The chain rule (one variable)
6. Functions of two variables: $f(x, y) = x^2 y^3$
7. Partial derivatives
8. The total derivative
9. Linear regression and gradient descent
10. The chain rule with more than one variable
11. Backpropagation in a tiny neural network

**Every plot opens in its own window**, with its sliders and buttons at the bottom of that window.
3D plots: drag with the left mouse button to rotate, right-drag to zoom, or press **spin**.

## Setup
Install once: `pip install numpy matplotlib PyQt6`, and on Ubuntu `sudo apt install libxcb-cursor0`. Then run this cell before anything else.

In [ ]:
# Plots open in their own windows (Qt, or Tk as a fallback).
# Qt is tested in a separate process first: if a system library is missing, Qt kills
# the whole Python process, and testing it here keeps the notebook kernel alive.
import subprocess, sys

def _works(test):
    try:
        r = subprocess.run([sys.executable, "-c", test], capture_output=True, text=True, timeout=30)
        return r.returncode == 0, (r.stderr.strip().splitlines() or ["(no message)"])[-1]
    except Exception as e:
        return False, str(e)

qt_ok, qt_msg = _works("from matplotlib.backends.qt_compat import QtWidgets; QtWidgets.QApplication([])")
if qt_ok:
    get_ipython().run_line_magic("matplotlib", "qt")
else:
    print("Qt can't open windows:", qt_msg)
    print("  Fix on Ubuntu:  sudo apt install libxcb-cursor0   (then restart the kernel)")
    tk_ok, tk_msg = _works("import tkinter; tkinter.Tk().destroy()")
    if not tk_ok:
        raise RuntimeError("No window backend available. Run in a terminal:\n"
                           "  sudo apt install libxcb-cursor0     (for Qt)\n"
                           "  sudo apt install python3-tk         (for Tk)\n"
                           "then restart the kernel and run this cell again.")
    print("Using Tk windows instead.")
    get_ipython().run_line_magic("matplotlib", "tk")

import time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button
from matplotlib.colors import TwoSlopeNorm

CLOSE_OTHERS = True   # opening a new plot window closes the previous one (one window on screen at a time)
MAXIMIZE = True       # open windows maximized (good for screen recording)

plt.rcParams.update({
    "figure.dpi": 100, "font.size": 13,
    "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2.5,
})
BLUE, ORANGE, GREEN, RED, PURPLE, GRAY = "#2563eb", "#ea580c", "#16a34a", "#dc2626", "#9333ea", "#6b7280"


# ------------------------------------------------------------------ windows
def _maximize(fig):
    win = fig.canvas.manager.window
    try:
        win.showMaximized()                  # Qt
    except AttributeError:
        try:
            win.attributes("-zoomed", True)  # Tk on Linux
        except Exception:
            pass


def window(name, size=(14, 8.5)):
    """Open (or reuse) a named plot window and clear it."""
    if CLOSE_OTHERS:
        for other in plt.get_figlabels():
            if other != name:
                plt.close(other)
    fig = plt.figure(name, figsize=size)
    for obj in getattr(fig, "_keep", []):
        if hasattr(obj, "stop"):
            obj.stop()                       # stop old animations when a cell is re-run
    fig.clf()
    fig._keep = []                           # widgets must stay referenced or they stop working
    if not getattr(fig, "_hooked", False):
        fig.canvas.mpl_connect("close_event",
                               lambda e: [o.stop() for o in fig._keep if hasattr(o, "stop")])
        fig._hooked = True
        if MAXIMIZE:
            _maximize(fig)
    return fig


def finish(fig):
    fig.canvas.draw_idle()
    try:
        fig.canvas.manager.show()            # bring the window to the front
    except Exception:
        pass
    plt.show(block=False)


def plot_bottom(n_controls):
    return 0.12 + 0.045 * n_controls


def panels(fig, n_controls, ncols=1, width_ratios=None, wspace=0.22, top=0.9):
    gs = fig.add_gridspec(1, ncols, left=0.06, right=0.98, top=top, bottom=plot_bottom(n_controls),
                          width_ratios=width_ratios, wspace=wspace)
    return [fig.add_subplot(gs[0, i]) for i in range(ncols)]


# ------------------------------------------------------------------ controls
def _row(fig, row, x=0.17, w=0.5, h=0.03):
    return fig.add_axes([x, 0.025 + row * 0.045, w, h])


def slider(fig, row, label, vmin, vmax, init, step=None, log=False, fmt="%.2f"):
    """A slider in the window. log=True: the slider moves the exponent, Δ = 10^value."""
    s = Slider(_row(fig, row), label, vmin, vmax, valinit=init, valstep=step, valfmt=fmt, color=BLUE)
    s.label.set_fontsize(13)
    s.valtext.set_fontsize(13)
    s.is_log = log
    if log:
        show = lambda v: s.valtext.set_text(f"{10 ** v:.1e}")
        show(init)
        s.on_changed(show)
    fig._keep.append(s)
    return s


def val(s):
    return 10 ** s.val if s.is_log else s.val


class Toggle:
    """An on/off button (big and easy to see on camera)."""
    def __init__(self, fig, row, label, init=False):
        ax = fig.add_axes([0.17, 0.021 + row * 0.045, 0.26, 0.037])
        self.button = Button(ax, "", color="#e5e7eb", hovercolor="#cbd5e1")
        self.button.label.set_fontsize(12)
        self.text, self.on, self._callbacks = label, init, []
        self.button.on_clicked(lambda event: self.set(not self.on))
        self._render()
        fig._keep.extend([self, self.button])

    def _render(self):
        self.button.label.set_text(("☑  " if self.on else "☐  ") + self.text)
        self.button.color = "#bfdbfe" if self.on else "#e5e7eb"
        self.button.ax.set_facecolor(self.button.color)

    def set(self, on):
        self.on = on
        self._render()
        for cb in self._callbacks:
            cb()

    def on_changed(self, cb):
        self._callbacks.append(cb)


def checkbox(fig, row, label, init=False):
    return Toggle(fig, row, label, init)


def checked(c):
    return c.on


def button(fig, row, label, on_click, col=0):
    ax = fig.add_axes([0.76 + col * 0.11, 0.021 + row * 0.045, 0.1, 0.037])
    b = Button(ax, label, color="#e5e7eb", hovercolor="#cbd5e1")
    b.label.set_fontsize(12)
    b.on_clicked(lambda event: on_click())
    fig._keep.append(b)
    return b


def live(draw, *controls):
    """Redraw whenever any control changes, and draw once now."""
    for c in controls:
        if isinstance(c, Slider):
            c.on_changed(lambda _: draw())
        elif isinstance(c, Toggle):
            c.on_changed(draw)
    draw()


# ------------------------------------------------------------------ animation
def every(fig, ms, step):
    """Call step() every ms milliseconds until it returns False."""
    t = fig.canvas.new_timer(interval=ms)
    def tick():
        if step() is False:
            t.stop()
    t.add_callback(tick)
    fig._keep.append(t)
    t.start()
    return t


def glide(fig, s, start, end, seconds=5.0):
    """Move a slider smoothly from start to end (the 'shrink Δx' moment on camera)."""
    if getattr(s, "_glide", None) is not None:
        s._glide.stop()
    t_start = time.perf_counter()
    def step():
        p = min(1.0, (time.perf_counter() - t_start) / seconds)   # finishes on time, even on a slow machine
        s.set_val(start + (end - start) * p)
        return p < 1.0
    s.set_val(start)
    s._glide = every(fig, 40, step)


def play_button(fig, row, s, vmax, ms=120, col=0, loop=False, label="▶ play"):
    """Play / pause button that steps slider s forward."""
    state = {"t": None}
    def stop():
        if state["t"] is not None:
            state["t"].stop()
        state["t"] = None
        b.label.set_text(label)
        fig.canvas.draw_idle()
    def toggle():
        if state["t"] is not None:
            return stop()
        if s.val >= vmax and not loop:
            s.set_val(s.valmin)
        def step():
            if s.val >= vmax:
                if loop:
                    s.set_val(s.valmin)
                    return True
                stop()
                return False
            s.set_val(min(s.val + (s.valstep or 1), vmax))
            return True
        state["t"] = every(fig, ms, step)
        b.label.set_text("|| pause")
        fig.canvas.draw_idle()
    b = button(fig, row, label, toggle, col)
    return b


def spin_button(fig, ax, row=0, col=0):
    """Rotate a 3D plot slowly (press again to stop)."""
    state = {"t": None}
    def toggle():
        if state["t"] is not None:
            state["t"].stop()
            state["t"] = None
            return
        def step():
            ax.view_init(ax.elev, ax.azim + 1.5)
            fig.canvas.draw_idle()
        state["t"] = every(fig, 50, step)
    return button(fig, row, "spin", toggle, col)


def axes3d(fig, rect=(0.02, 0.09, 0.96, 0.83)):
    ax = fig.add_axes(rect, projection="3d")
    try:
        ax.computed_zorder = False           # draw lines on top of the surface
    except Exception:
        pass
    return ax


# ------------------------------------------------------------------ math helpers
def line_eq(m, c):
    return f"y = {m:.3g}x {'+' if c >= 0 else '-'} {abs(c):.3g}"


def slope_table(f, x0, exact=None, dxs=(1, 0.1, 0.01, 0.001, 1e-4, 1e-5, 1e-6), name="x", out="f"):
    """Print (f(x0 + d) - f(x0)) / d for smaller and smaller steps d."""
    head = f"{'Δ' + name:>10} | {'Δ' + out:>16} | {'slope Δ' + out + '/Δ' + name:>16}"
    if exact is not None:
        head += f" | {'error':>10}"
    print(head)
    print("-" * len(head))
    for d in dxs:
        df = f(x0 + d) - f(x0)
        s = df / d
        row = f"{d:>10g} | {df:>16.10f} | {s:>16.10f}"
        if exact is not None:
            row += f" | {abs(s - exact):>10.1e}"
        print(row)
    if exact is not None:
        print(f"\nexact answer = {exact}")


class Secant:
    """A curve with two points, the run/rise triangle and the line through them.
    The lines are created once and only moved afterwards, so sliders stay smooth."""
    def __init__(self, ax, f, xs, label_f="f(x)", dx_name="x", df_name="f", curve_color=BLUE):
        self.ax, self.f, self.xs, self.dx_name, self.df_name = ax, f, xs, dx_name, df_name
        self.curve, = ax.plot(xs, f(xs), color=curve_color, label=label_f)
        self.line, = ax.plot([], [], "--", color=ORANGE, lw=2)
        self.tangent, = ax.plot([], [], color=PURPLE, lw=1.8, alpha=0.9, visible=False)
        self.run, = ax.plot([], [], color=GREEN, lw=2.5)
        self.rise, = ax.plot([], [], color=RED, lw=2.5)
        self.pts, = ax.plot([], [], "o", color=ORANGE, ms=9, zorder=5)
        self.t_run = ax.annotate("", (0, 0), textcoords="offset points", xytext=(0, -20), ha="center",
                                 color=GREEN, fontsize=12, annotation_clip=False)
        self.t_rise = ax.annotate("", (0, 0), textcoords="offset points", xytext=(8, 0),
                                  color=RED, fontsize=12, annotation_clip=False)
        self.slope = None

    def set_function(self, f, label):
        self.f = f
        self.curve.set_ydata(f(self.xs))
        self.curve.set_label(label)

    def update(self, x1, x2):
        """Move the two points to x1 and x2. Returns the slope (None if x1 == x2)."""
        f, xs = self.f, self.xs
        y1, y2 = f(x1), f(x2)
        self.pts.set_data([x1, x2], [y1, y2])
        if abs(x2 - x1) < 1e-9:
            for a in (self.line, self.run, self.rise):
                a.set_data([], [])
            self.t_run.set_visible(False); self.t_rise.set_visible(False)
            self.slope = None
            return None
        m = (y2 - y1) / (x2 - x1)
        self.line.set_data(xs, y1 + m * (xs - x1))
        self.line.set_label(f"line through the 2 points, slope = {m:.4f}")
        self.run.set_data([x1, x2], [y1, y1])
        self.rise.set_data([x2, x2], [y1, y2])
        big = abs(x2 - x1) >= 0.04 * (np.nanmax(xs) - np.nanmin(xs))   # label only when there is room
        self.t_run.xy = ((x1 + x2) / 2, y1); self.t_run.set_text(f"Δ{self.dx_name} = {x2 - x1:.3g}")
        self.t_rise.xy = (x2, (y1 + y2) / 2); self.t_rise.set_text(f"Δ{self.df_name} = {y2 - y1:.3g}")
        self.t_run.set_visible(big); self.t_rise.set_visible(big)
        self.slope = m
        return m

    def show_tangent(self, x0, slope, visible=True):
        self.tangent.set_data(self.xs, self.f(x0) + slope * (self.xs - x0))
        self.tangent.set_label(f"tangent line, slope = {slope:.4f}")
        self.tangent.set_visible(visible)

    def legend(self, **kw):
        handles = [self.curve] + ([self.line] if self.slope is not None else []) \
                  + ([self.tangent] if self.tangent.get_visible() else [])
        self.ax.legend(handles=handles, **kw)


def limit_window(name, f, exact_slope, xs, xlim, ylim, label_f, x0_range, x0_default):
    """Left: the curve with a secant from x0 to x0 + Δx.  Right: the slope as Δx shrinks."""
    fig = window(name, (16, 8.5))
    ax, ax2 = panels(fig, 3, ncols=2, width_ratios=[1.3, 1])
    x0 = slider(fig, 2, "x0", x0_range[0], x0_range[1], x0_default, step=0.05)
    dx = slider(fig, 1, "Δx", -6, 0, 0, log=True)
    tan = checkbox(fig, 0, "show tangent line")
    button(fig, 1, "▶ shrink Δx", lambda: glide(fig, dx, 0, -6, 6))
    button(fig, 1, "reset Δx", lambda: dx.set_val(0), col=1)

    sec = Secant(ax, f, xs, label_f=label_f)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_xlabel("x"); ax.set_ylabel("f(x)")
    d_all = np.logspace(-6, 0, 300)
    curve, = ax2.semilogx(d_all, d_all, color=ORANGE)
    target = ax2.axhline(0, color=PURPLE, ls=":", lw=2)
    dot, = ax2.plot([], [], "o", color=RED, ms=11, zorder=5)
    ax2.set_xlim(1.5, 5e-7)                      # reversed: Δx gets smaller to the right
    ax2.set_xlabel("Δx   (smaller and smaller  →)"); ax2.set_ylabel("slope Δf / Δx")
    ax2.set_title("the slope as Δx → 0")
    last_x0 = [None]

    def draw():
        a, d = x0.val, val(dx)
        exact = exact_slope(a)
        m = sec.update(a, a + d)
        sec.show_tangent(a, exact, checked(tan))
        sec.legend(loc="upper left", fontsize=11)
        ax.set_title(f"x0 = {a:.2f}    Δx = {d:.2g}    slope ≈ {m:.6f}")
        if last_x0[0] != a:                      # the right panel only changes when x0 moves
            ys = (f(a + d_all) - f(a)) / d_all
            curve.set_ydata(ys)
            target.set_ydata([exact, exact])
            target.set_label(f"the value it approaches: {exact:.4f}")
            lo, hi = min(ys.min(), exact), max(ys.max(), exact)
            pad = 0.08 * (hi - lo or 1)
            ax2.set_ylim(lo - pad, hi + pad)
            ax2.legend(handles=[target], loc="best")
            last_x0[0] = a
        dot.set_data([d], [m])
        fig.canvas.draw_idle()

    live(draw, x0, dx, tan)
    finish(fig)

print("Ready. Plots open in windows using:", matplotlib.get_backend())

---
# 1. The slope of a straight line

$$f(x) = 2x + 1$$

In [ ]:
def f_line(x):
    return 2 * x + 1

xs1 = np.linspace(-1, 5, 200)
fig = window("1. A straight line", (12, 8))
ax = fig.add_subplot()
ax.plot(xs1, f_line(xs1), color=BLUE, label="f(x) = 2x + 1")
ax.axhline(0, color="k", lw=0.8); ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("x"); ax.set_ylabel("f(x)"); ax.set_title("A straight line"); ax.legend()
finish(fig)

Pick **any two points** on the line. Go right by $\Delta x$, go up by $\Delta f$:

$$\text{slope} = \frac{\text{rise}}{\text{run}} = \frac{\Delta f}{\Delta x} = \frac{f(x_2) - f(x_1)}{x_2 - x_1}$$

In the window: drag **x1** and **x2**. The button **x1 = x2** puts both points on top of each other.

In [ ]:
def scene_1_slope():
    fig = window("1. Slope of a line", (13, 8.5))
    ax, = panels(fig, 2)
    x1 = slider(fig, 1, "x1", -1, 5, 1, step=0.1)
    x2 = slider(fig, 0, "x2", -1, 5, 3, step=0.1)
    button(fig, 1, "x1 = x2", lambda: x1.set_val(x2.val))
    sec = Secant(ax, f_line, xs1, label_f="f(x) = 2x + 1")
    ax.set_xlim(-1, 5); ax.set_ylim(-2, 12); ax.set_xlabel("x"); ax.set_ylabel("f(x)")

    def draw():
        a, b = x1.val, x2.val
        m = sec.update(a, b)
        if m is None:
            ax.set_title("Δx = 0   →   slope = 0 / 0 = ???", color=RED)
        else:
            ax.set_title(f"slope = Δf / Δx = {f_line(b) - f_line(a):.2f} / {b - a:.2f} = {m:.2f}", color="k")
        sec.legend(loc="upper left")
        fig.canvas.draw_idle()

    live(draw, x1, x2)
    finish(fig)

scene_1_slope()

---
# 2. A curve: $f(x) = x^2$

Same recipe, two points and $\dfrac{\Delta f}{\Delta x}$. But now the answer depends on **which** points you pick.

In [ ]:
def f_sq(x):
    return x ** 2

xs2 = np.linspace(-1, 3.5, 300)

def scene_2_points():
    fig = window("2. Two points on x²", (13, 8.5))
    ax, = panels(fig, 2)
    p1 = slider(fig, 1, "x1", -1, 3.5, 0.5, step=0.1)
    p2 = slider(fig, 0, "x2", -1, 3.5, 2.5, step=0.1)
    sec = Secant(ax, f_sq, xs2, label_f="f(x) = x²")
    ax.set_xlim(-1, 3.5); ax.set_ylim(-1, 12); ax.set_xlabel("x"); ax.set_ylabel("f(x)")

    def draw():
        m = sec.update(p1.val, p2.val)
        ax.set_title("Δx = 0  →  0 / 0" if m is None else f"slope between the two points = {m:.3f}")
        sec.legend(loc="upper left")
        fig.canvas.draw_idle()

    live(draw, p1, p2)
    finish(fig)

scene_2_points()

### Bring the second point closer and closer

Fix the first point at $x_0$ and put the second one at $x_0 + \Delta x$. Now shrink $\Delta x$
(drag the **Δx** slider, or press **▶ shrink Δx**):

$$\frac{dy}{dx} = \lim_{\Delta x \to 0} \frac{f(x_0 + \Delta x) - f(x_0)}{\Delta x}$$

In [ ]:
limit_window("2. Shrinking Δx on x²", f_sq, exact_slope=lambda x0: 2 * x0, xs=xs2,
             xlim=(-1, 3.5), ylim=(-1, 12), label_f="f(x) = x²", x0_range=(-0.5, 2.5), x0_default=1.0)

The same thing as numbers, at $x_0 = 1$ (from the right, then from the left):

In [ ]:
slope_table(f_sq, x0=1.0, exact=2.0)
print("\nfrom the left (negative Δx):\n")
slope_table(f_sq, x0=1.0, exact=2.0, dxs=(-1, -0.1, -0.01, -0.001, -1e-4, -1e-5, -1e-6))

### Zoom in: every smooth curve looks like a straight line up close

In [ ]:
def scene_2_zoom():
    fig = window("2. Zooming in on x²", (11, 9))
    ax, = panels(fig, 1)
    zoom = slider(fig, 0, "zoom (10^k)", 0, 3, 0, step=0.05)
    button(fig, 0, "▶ zoom in", lambda: glide(fig, zoom, 0, 3, 5))
    x0 = 1.0
    curve, = ax.plot([], [], color=BLUE, lw=3, label="f(x) = x²")
    line, = ax.plot([], [], "--", color=PURPLE, lw=1.8, label="line with slope 2")
    ax.plot(x0, f_sq(x0), "o", color=RED, ms=9)
    ax.legend(loc="upper left")

    def draw():
        half = 1.5 / 10 ** zoom.val
        xs = np.linspace(x0 - half, x0 + half, 400)
        curve.set_data(xs, f_sq(xs))
        line.set_data(xs, f_sq(x0) + 2 * x0 * (xs - x0))
        ys = np.concatenate([f_sq(xs), f_sq(x0) + 2 * x0 * (xs - x0)])
        pad = 0.05 * (ys.max() - ys.min())
        ax.set_xlim(xs[0], xs[-1]); ax.set_ylim(ys.min() - pad, ys.max() + pad)
        ax.set_title(f"zoom × {10 ** zoom.val:,.0f}  around x = 1")
        fig.canvas.draw_idle()

    live(draw, zoom)
    finish(fig)

scene_2_zoom()

### Do it at every $x$: the slope becomes a new function

In [ ]:
xs = np.linspace(-3, 3, 400)
h = 1e-5
measured = (f_sq(xs + h) - f_sq(xs)) / h

fig = window("2. The derivative of x²", (16, 7))
a1, a2 = panels(fig, 0, ncols=2)
a1.plot(xs, f_sq(xs), color=BLUE); a1.set_title("f(x) = x²")
a2.plot(xs, measured, color=ORANGE, lw=5, alpha=0.5, label="measured slope Δf/Δx  (Δx = 0.00001)")
a2.plot(xs, 2 * xs, "--", color=PURPLE, label="2x")
a2.set_title("slope of x² at every point:  f'(x) = 2x"); a2.legend()
for ax in (a1, a2):
    ax.axhline(0, color="k", lw=0.8); ax.set_xlabel("x")
finish(fig)

### Bonus: why can't the computer just use a really, really small $\Delta x$?

In [ ]:
slope_table(f_sq, x0=1.0, exact=2.0, dxs=(1e-6, 1e-8, 1e-10, 1e-12, 1e-14, 1e-15, 1e-16))
print("\n1 + 1e-16 == 1 ?", 1 + 1e-16 == 1)

---
# 3. Velocity = change of distance over time

A car leaves a traffic light and stops at the next one, 32 m away, 4 seconds later:

$$s(t) = 6t^2 - t^3 \quad\text{(meters)}$$

$$\text{average velocity} = \frac{\Delta s}{\Delta t}, \qquad \text{velocity right now} = \lim_{\Delta t \to 0}\frac{\Delta s}{\Delta t} = \frac{ds}{dt}$$

In [ ]:
def s_car(t):
    return 6 * t ** 2 - t ** 3        # distance in meters, t in seconds

def v_car(t):
    return 12 * t - 3 * t ** 2        # the speedometer (we "discover" this by shrinking Δt)

def scene_3_velocity():
    ts = np.linspace(0, 4, 300)

    fig = window("3. Velocity", (16, 9.5))
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 5], hspace=0.5, wspace=0.2,
                          left=0.06, right=0.98, top=0.93, bottom=plot_bottom(3))
    road, a1, a2 = fig.add_subplot(gs[0, :]), fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])
    t0 = slider(fig, 2, "t0 (s)", 0, 3.9, 1.0, step=0.05)
    dt = slider(fig, 1, "Δt (s)", -5, 0, 0, log=True)
    sp = checkbox(fig, 0, "show the speedometer")
    button(fig, 1, "▶ shrink Δt", lambda: glide(fig, dt, 0, -5, 5))
    button(fig, 1, "reset Δt", lambda: dt.set_val(0), col=1)

    road.plot([0, 33], [0, 0], color=GRAY, lw=8, solid_capstyle="butt")
    car_now, = road.plot([], [], ">", ms=20, color=BLUE)
    car_later, = road.plot([], [], ">", ms=20, color=ORANGE, alpha=0.6)
    road.set_xlim(-1, 34); road.set_ylim(-1, 1); road.set_yticks([]); road.grid(False)
    road.set_xlabel("position on the road (m)")

    sec = Secant(a1, s_car, ts, label_f="distance s(t)", dx_name="t", df_name="s")
    a1.set_xlim(0, 4.2); a1.set_ylim(-2, 36); a1.set_xlabel("time t (s)"); a1.set_ylabel("distance (m)")

    speedo, = a2.plot(ts, v_car(ts), color=GREEN, label="speedometer v(t)")
    true_v, = a2.plot([], [], "x", color=RED, ms=14, mew=3)
    estimate, = a2.plot([], [], "o", color=ORANGE, ms=11)
    a2.set_xlim(0, 4.2); a2.set_ylim(-3, 15); a2.set_xlabel("time t (s)"); a2.set_ylabel("velocity (m/s)")
    a2.set_title("velocity")

    def draw():
        t, d = t0.val, val(dt)
        car_now.set_data([s_car(t)], [0]); car_later.set_data([s_car(t + d)], [0])
        road.set_title(f"car at t = {t:.2f} s (blue) and at t = {t + d:.4g} s (orange)")
        avg = sec.update(t, t + d)
        sec.legend(loc="upper left", fontsize=10)
        a1.set_title(f"average velocity  Δs/Δt = {avg:.4f} m/s")
        on = checked(sp)
        speedo.set_visible(on); true_v.set_visible(on)
        true_v.set_data([t], [v_car(t)]); true_v.set_label(f"v({t:.2f}) = {v_car(t):.4f} m/s")
        estimate.set_data([t], [avg]); estimate.set_label(f"our estimate Δs/Δt = {avg:.4f}")
        a2.legend(handles=([speedo, true_v] if on else []) + [estimate], loc="lower center", fontsize=10)
        fig.canvas.draw_idle()

    live(draw, t0, dt, sp)
    finish(fig)

scene_3_velocity()

In [ ]:
# how fast is the car going at exactly t = 2 s ?
slope_table(s_car, x0=2.0, exact=12.0, name="t", out="s")

---
# 4. $f(x) = \dfrac{1}{x}$

In [ ]:
def f_inv(x):
    return 1 / x

xl = np.linspace(-4, -0.2, 300); xr = np.linspace(0.2, 4, 300)
fig = window("4. f(x) = 1/x", (12, 8))
ax = fig.add_subplot()
ax.plot(xl, f_inv(xl), color=BLUE); ax.plot(xr, f_inv(xr), color=BLUE, label="f(x) = 1/x")
ax.axhline(0, color="k", lw=0.8); ax.axvline(0, color="k", lw=0.8)
ax.set_ylim(-5, 5); ax.set_xlabel("x"); ax.set_title("f(x) = 1/x"); ax.legend()
finish(fig)

In [ ]:
limit_window("4. Shrinking Δx on 1/x", f_inv, exact_slope=lambda x0: -1 / x0 ** 2, xs=np.linspace(0.15, 5, 400),
             xlim=(0, 5), ylim=(-1, 5), label_f="f(x) = 1/x", x0_range=(0.3, 4), x0_default=2.0)

In [ ]:
slope_table(f_inv, x0=2.0, exact=-0.25)

In [ ]:
xs = np.concatenate([np.linspace(-3, -0.3, 200), [np.nan], np.linspace(0.3, 3, 200)])
h = 1e-6
fig = window("4. The derivative of 1/x", (16, 7))
a1, a2 = panels(fig, 0, ncols=2)
a1.plot(xs, f_inv(xs), color=BLUE); a1.set_title("f(x) = 1/x"); a1.set_ylim(-4, 4)
a2.plot(xs, (f_inv(xs + h) - f_inv(xs)) / h, color=ORANGE, lw=5, alpha=0.5, label="measured slope")
a2.plot(xs, -1 / xs ** 2, "--", color=PURPLE, label="-1/x²")
a2.set_ylim(-12, 1); a2.set_title("slope of 1/x:  f'(x) = -1/x²  (always negative)"); a2.legend()
for ax in (a1, a2):
    ax.axhline(0, color="k", lw=0.8); ax.axvline(0, color="k", lw=0.8); ax.set_xlabel("x")
finish(fig)

---
# 5. The chain rule (one variable)

The wind power at time $t$ is $u = f(t) = 2t^2$.
The turbine turns that power into output $g(u) = u^2 + 5$.

How fast does the output change with **time**? A nudge $\Delta t$ causes a nudge $\Delta u$, which causes a nudge $\Delta g$:

$$\frac{\Delta g}{\Delta t} = \frac{\Delta g}{\Delta u}\cdot\frac{\Delta u}{\Delta t} \quad\xrightarrow{\;\Delta t \to 0\;}\quad \frac{dg}{dt} = \frac{dg}{du}\cdot\frac{du}{dt} = g'(f(t))\cdot f'(t)$$

In [ ]:
def wind(t):          # u = f(t): wind power at time t
    return 2 * t ** 2

def turbine(u):       # g(u): output for a given wind power
    return u ** 2 + 5

def output(t):        # g(f(t)) = 4t^4 + 5
    return turbine(wind(t))

def scene_5_chain():
    fig = window("5. The chain rule", (18, 8.5))
    a1, a2, a3 = panels(fig, 2, ncols=3, top=0.8)
    t0 = slider(fig, 1, "t0", 0.1, 1.2, 1.0, step=0.05)
    dt = slider(fig, 0, "Δt", -5, -0.4, np.log10(0.3), log=True)
    button(fig, 0, "▶ shrink Δt", lambda: glide(fig, dt, -0.4, -5, 5))
    ts5, us5 = np.linspace(0, 1.6, 300), np.linspace(0, 5.2, 300)

    s1 = Secant(a1, wind, ts5, label_f="u = f(t) = 2t²", dx_name="t", df_name="u")
    s2 = Secant(a2, turbine, us5, label_f="g(u) = u² + 5", dx_name="u", df_name="g", curve_color=GREEN)
    s3 = Secant(a3, output, ts5, label_f="g(f(t))", dx_name="t", df_name="g", curve_color=PURPLE)
    a1.set_ylim(-0.5, 5.5); a1.set_xlabel("t"); a1.set_ylabel("u")
    a2.set_ylim(0, 32); a2.set_xlabel("u"); a2.set_ylabel("g")
    a3.set_ylim(0, 32); a3.set_xlabel("t"); a3.set_ylabel("g")
    for ax, xs in ((a1, ts5), (a2, us5), (a3, ts5)):
        ax.set_xlim(xs[0] - 0.05 * xs[-1], xs[-1] * 1.05)

    def draw():
        t, d = t0.val, val(dt)
        u0, u1 = wind(t), wind(t + d)
        r1 = s1.update(t, t + d)
        r2 = s2.update(u0, u1)
        r3 = s3.update(t, t + d)
        a1.set_title(f"time → wind power\nΔu/Δt = {r1:.4f}")
        a2.set_title(f"wind power → output\nΔg/Δu = {r2:.4f}")
        a3.set_title(f"time → output\nΔg/Δt = {r3:.4f}")
        for sec in (s1, s2, s3):
            sec.legend(loc="upper left", fontsize=9)
        fig.suptitle(f"(Δg/Δu)·(Δu/Δt) = {r2:.4f} × {r1:.4f} = {r2 * r1:.4f}      "
                     f"exact:  g'(f(t0))·f'(t0) = {2 * u0:.3f} × {4 * t:.3f} = {2 * u0 * 4 * t:.4f}", fontsize=15)
        fig.canvas.draw_idle()

    live(draw, t0, dt)
    finish(fig)

scene_5_chain()

In [ ]:
t0 = 1.0
print(f"{'Δt':>8} | {'Δu/Δt':>12} | {'Δg/Δu':>12} | {'product':>12} | {'Δg/Δt':>12}")
print("-" * 68)
for dt in (0.1, 0.01, 0.001, 1e-4, 1e-5, 1e-6):
    du = wind(t0 + dt) - wind(t0)
    dg = turbine(wind(t0) + du) - turbine(wind(t0))
    print(f"{dt:>8g} | {du / dt:>12.6f} | {dg / du:>12.6f} | {dg / du * du / dt:>12.6f} | {dg / dt:>12.6f}")
print(f"\nchain rule:  g'(f(1)) · f'(1) = (2·u) · (4t) = {2 * wind(t0)} × {4 * t0} = {2 * wind(t0) * 4 * t0}")

### Now the version from the plan: $g = f(t)^2 \cdot t + 5$

Here $t$ shows up **twice**: inside $f(t)$ and on its own. Watch what the one-path chain rule gives:

In [ ]:
def g_two_paths(t):
    return wind(t) ** 2 * t + 5            # = 4t^5 + 5

t0, h = 1.0, 1e-6
u0 = wind(t0)
measured = (g_two_paths(t0 + h) - g_two_paths(t0)) / h
through_u_only = (2 * u0 * t0) * (4 * t0)   # ∂g/∂u · du/dt   (only the path through u)
direct_t = u0 ** 2                          # ∂g/∂t           (the direct path through t)

print(f"measured slope at t = 1        : {measured:.4f}")
print(f"chain rule through u only      : {through_u_only:.4f}   <- something is missing!")
print(f"the missing piece  ∂g/∂t = u²  : {direct_t:.4f}")
print(f"sum of both paths              : {through_u_only + direct_t:.4f}   (we come back to this in part 10)")

---
# 6. A function of two variables

$$f(x, y) = x^2 y^3$$

Two inputs, one output, so the graph is a **surface**. Drag with the mouse to rotate (right-drag zooms), or press **spin**.

In [ ]:
def F(x, y):
    return x ** 2 * y ** 3

X, Y = np.meshgrid(np.linspace(-2, 2, 50), np.linspace(-1.5, 1.5, 50))
Z = F(X, Y)
NORM = TwoSlopeNorm(vmin=Z.min(), vcenter=0, vmax=Z.max())
ZLIM = (Z.min(), Z.max())

def label3d(ax, z="f(x, y)"):
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel(z)
    ax.view_init(elev=22, azim=-50)

fig = window("6. f(x, y) = x² y³", (13, 9.5))
ax = axes3d(fig, (0.0, 0.08, 0.92, 0.88))
surf = ax.plot_surface(X, Y, Z, cmap="RdBu_r", norm=NORM, linewidth=0, antialiased=True)
fig.colorbar(surf, ax=ax, shrink=0.6, pad=0.06, label="f")
ax.set_title("f(x, y) = x² y³")
label3d(ax)
spin_button(fig, ax)
finish(fig)

---
# 7. Partial derivatives: freeze one variable, nudge the other

Freeze $y = y_0$. Now $f(x, y_0)$ is an ordinary one-variable function of $x$, and we already know what to do:

$$\frac{\partial f}{\partial x} = \lim_{\Delta x \to 0} \frac{f(x_0 + \Delta x,\, y_0) - f(x_0,\, y_0)}{\Delta x} \qquad
\frac{\partial f}{\partial y} = \lim_{\Delta y \to 0} \frac{f(x_0,\, y_0 + \Delta y) - f(x_0,\, y_0)}{\Delta y}$$

In the window: the gray wall is the frozen variable, the black curve is the slice. Shrink the step with the slider or **▶ shrink**.

In [ ]:
def partial_window(wrt="x", x0=1.0, y0=1.0):
    fig = window(f"7. Partial derivative ∂f/∂{wrt}", (13, 10))
    ax = axes3d(fig, (0.02, 0.13, 0.96, 0.78))
    ax.plot_surface(X, Y, Z, cmap="RdBu_r", norm=NORM, alpha=0.45, linewidth=0)
    z0 = F(x0, y0)
    w = np.linspace(*ZLIM, 2)
    if wrt == "x":
        s = np.linspace(-2, 2, 200)
        cx, cy = s, np.full_like(s, y0)
        P, W = np.meshgrid(np.linspace(-2, 2, 2), w)
        ax.plot_surface(P, np.full_like(P, y0), W, color="gray", alpha=0.18, linewidth=0)
        exact = 2 * x0 * y0 ** 3
        second = lambda d: (x0 + d, y0)
        along = lambda tau: (x0 + tau, np.full_like(tau, y0))
        frozen = f"y frozen at {y0}"
    else:
        s = np.linspace(-1.5, 1.5, 200)
        cx, cy = np.full_like(s, x0), s
        P, W = np.meshgrid(np.linspace(-1.5, 1.5, 2), w)
        ax.plot_surface(np.full_like(P, x0), P, W, color="gray", alpha=0.18, linewidth=0)
        exact = 3 * x0 ** 2 * y0 ** 2
        second = lambda d: (x0, y0 + d)
        along = lambda tau: (np.full_like(tau, x0), y0 + tau)
        frozen = f"x frozen at {x0}"

    ax.plot(cx, cy, F(cx, cy), color="k", lw=3, zorder=10, label="the slice")
    ax.plot([x0], [y0], [z0], "o", color=RED, ms=9, zorder=12)
    sec, = ax.plot([], [], [], color=ORANGE, lw=3.5, zorder=11)
    pt, = ax.plot([], [], [], "o", color=ORANGE, ms=8, zorder=12)
    tan, = ax.plot([], [], [], color=PURPLE, lw=2.5, zorder=11)
    ax.set_zlim(*ZLIM)
    label3d(ax)

    def clip(tau, m):
        z = z0 + m * tau
        z = np.where((z >= ZLIM[0]) & (z <= ZLIM[1]), z, np.nan)
        lx, ly = along(tau)
        return lx, ly, z

    step = slider(fig, 1, f"Δ{wrt}", -3, 0, 0, log=True)
    show_tan = checkbox(fig, 0, "show tangent (exact slope)")
    button(fig, 1, "▶ shrink", lambda: glide(fig, step, 0, -3, 5))
    button(fig, 1, "reset", lambda: step.set_val(0), col=1)
    spin_button(fig, ax, row=0)
    name = "∂f/∂x" if wrt == "x" else "∂f/∂y"

    def draw():
        d = val(step)
        px, py = second(d)
        m = (F(px, py) - z0) / d
        sec.set_data_3d(*clip(np.linspace(-0.9, 0.9 + d, 60), m))
        pt.set_data_3d([px], [py], [F(px, py)])
        tan.set_visible(checked(show_tan))
        tan.set_data_3d(*clip(np.linspace(-0.9, 0.9, 60), exact))
        ax.set_title(f"{name} at ({x0}, {y0}),  {frozen}\nΔ{wrt} = {d:.3g}   →   slope = {m:.6f}   (exact {exact:.4f})")
        fig.canvas.draw_idle()

    live(draw, step, show_tan)
    finish(fig)

partial_window("x")

In [ ]:
partial_window("y")

### The same slice, flattened to 2D (move the frozen value around)

In [ ]:
def scene_7_slice():
    fig = window("7. The slice in 2D", (13, 9))
    ax, = panels(fig, 3)
    fy = slider(fig, 2, "frozen y", -1.5, 1.5, 1.0, step=0.05)
    x0 = slider(fig, 1, "x0", -1.5, 1.5, 1.0, step=0.05)
    dx = slider(fig, 0, "Δx", -5, 0, 0, log=True)
    button(fig, 0, "▶ shrink Δx", lambda: glide(fig, dx, 0, -5, 5))
    sec = Secant(ax, lambda x: F(x, 1.0), np.linspace(-2, 2.5, 300))
    ax.set_xlim(-2.2, 2.7); ax.set_ylim(-14, 14); ax.set_xlabel("x")

    def draw():
        yv, a, d = fy.val, x0.val, val(dx)
        sec.set_function(lambda x: F(x, yv), f"f(x, {yv:.2f}) = x² · ({yv:.2f})³")   # y is just a number now
        m = sec.update(a, a + d)
        exact = 2 * a * yv ** 3
        sec.legend(loc="upper left", fontsize=11)
        ax.set_title(f"y frozen at {yv:.2f}:   Δf/Δx = {m:.5f}    exact ∂f/∂x = 2·x·y³ = {exact:.5f}")
        fig.canvas.draw_idle()

    live(draw, fy, x0, dx)
    finish(fig)

scene_7_slice()

In [ ]:
x0, y0 = 1.0, 1.0
print("∂f/∂x at (1, 1): nudge x, keep y fixed\n")
slope_table(lambda x: F(x, y0), x0, exact=2 * x0 * y0 ** 3)
print("\n\n∂f/∂y at (1, 1): nudge y, keep x fixed\n")
slope_table(lambda y: F(x0, y), y0, exact=3 * x0 ** 2 * y0 ** 2, name="y")

---
# 8. The total derivative: nudge both at once

Move $x$ by $\Delta x$ **and** $y$ by $\Delta y$. For small steps the change in $f$ is

$$\Delta f \;\approx\; \frac{\partial f}{\partial x}\,\Delta x + \frac{\partial f}{\partial y}\,\Delta y
\qquad\Longrightarrow\qquad df = \frac{\partial f}{\partial x}\,dx + \frac{\partial f}{\partial y}\,dy$$

Geometrically: zoom in far enough and the surface looks like a flat **tangent plane**. The two partial derivatives are its two slopes.
The vector of both slopes is the **gradient**: $\nabla f = \left(\dfrac{\partial f}{\partial x}, \dfrac{\partial f}{\partial y}\right)$.

In [ ]:
x0, y0 = 1.0, 1.0
fx, fy = 2 * x0 * y0 ** 3, 3 * x0 ** 2 * y0 ** 2
Xp, Yp = np.meshgrid(np.linspace(x0 - 0.7, x0 + 0.7, 10), np.linspace(y0 - 0.5, y0 + 0.5, 10))
Zp = F(x0, y0) + fx * (Xp - x0) + fy * (Yp - y0)

fig = window("8. Tangent plane", (13, 9.5))
ax = axes3d(fig)
ax.plot_surface(X, Y, Z, cmap="RdBu_r", norm=NORM, alpha=0.6, linewidth=0)
ax.plot_surface(Xp, Yp, Zp, color=GREEN, alpha=0.85, linewidth=0)
ax.plot([x0], [y0], [F(x0, y0)], "o", color=RED, ms=10, zorder=12)
ax.set_title(f"tangent plane at (1, 1):  z = 1 + {fx:g}(x - 1) + {fy:g}(y - 1)")
label3d(ax)
spin_button(fig, ax)
finish(fig)

In [ ]:
print(f"{'step':>7} | {'Δx':>7} | {'Δy':>7} | {'real Δf':>12} | {'fx·Δx + fy·Δy':>14} | {'error':>10} | {'error/step':>10}")
print("-" * 86)
for h in (0.5, 0.1, 0.01, 0.001, 1e-4):
    dx, dy = 0.6 * h, 0.8 * h                     # walk diagonally, both variables change
    real = F(x0 + dx, y0 + dy) - F(x0, y0)
    linear = fx * dx + fy * dy
    print(f"{h:>7g} | {dx:>7.4g} | {dy:>7.4g} | {real:>12.8f} | {linear:>14.8f} | {abs(real - linear):>10.2e} | {abs(real - linear) / h:>10.2e}")
print("\nThe error shrinks much faster than the step itself: that is what 'the plane is the best linear approximation' means.")

---
# 9. Linear regression and gradient descent

We have data points and want the line $\hat y = m x + c$ that fits them best.
The line has two knobs, $m$ and $c$. How bad a line is gets measured by the **loss** (mean squared error):

$$L(m, c) = \frac{1}{n}\sum_{i=1}^{n}\big(m x_i + c - y_i\big)^2$$

$L$ is a function of **two variables** with a surface, just like $f(x, y)$. Its partial derivatives are

$$\frac{\partial L}{\partial m} = \frac{2}{n}\sum_i (m x_i + c - y_i)\,x_i \qquad \frac{\partial L}{\partial c} = \frac{2}{n}\sum_i (m x_i + c - y_i)$$

Gradient descent: take a small step **downhill**, against the gradient, and repeat:

$$m \leftarrow m - \eta\,\frac{\partial L}{\partial m} \qquad c \leftarrow c - \eta\,\frac{\partial L}{\partial c}$$

In [ ]:
rng = np.random.default_rng(7)
x_data = np.sort(rng.uniform(-3, 3, 25))
y_data = 1.8 * x_data + 2 + rng.normal(0, 0.8, 25)      # hidden "true" line: y = 1.8x + 2, plus noise

def loss(m, c):
    return np.mean((m * x_data + c - y_data) ** 2)

def gradient(m, c):
    r = m * x_data + c - y_data                          # residual = prediction - truth
    return 2 * np.mean(r * x_data), 2 * np.mean(r)       # (∂L/∂m, ∂L/∂c)

m_best, c_best = np.polyfit(x_data, y_data, 1)           # the answer, only used to draw a star

m0, c0 = -1.5, -3.0
fig = window("9. The data", (12, 8))
ax = fig.add_subplot()
ax.scatter(x_data, y_data, color=BLUE, zorder=3, label=f"{len(x_data)} data points")
xl = np.array([-3.3, 3.3])
ax.plot(xl, m0 * xl + c0, color=ORANGE, label=f"starting guess: {line_eq(m0, c0)}")
ax.set_title(f"starting loss L({m0}, {c0}) = {loss(m0, c0):.2f}"); ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend()
finish(fig)

In [ ]:
# Check the formulas for the partial derivatives with the "tiny nudge" method from part 7
h = 1e-6
gm, gc = gradient(m0, c0)
print(f"∂L/∂m   formula: {gm:.6f}    nudge m: {(loss(m0 + h, c0) - loss(m0, c0)) / h:.6f}")
print(f"∂L/∂c   formula: {gc:.6f}    nudge c: {(loss(m0, c0 + h) - loss(m0, c0)) / h:.6f}")

In [ ]:
def gradient_descent(m0, c0, lr, steps):
    path = [(m0, c0)]
    m, c = m0, c0
    for _ in range(steps):
        gm, gc = gradient(m, c)
        m, c = m - lr * gm, c - lr * gc
        path.append((m, c))
    return np.array(path)

# the first few steps, by the numbers
m, c, lr = m0, c0, 0.05
print(f"{'step':>4} | {'m':>8} | {'c':>8} | {'loss':>9} | {'∂L/∂m':>9} | {'∂L/∂c':>9}")
print("-" * 60)
for k in range(6):
    gm, gc = gradient(m, c)
    print(f"{k:>4} | {m:>8.4f} | {c:>8.4f} | {loss(m, c):>9.4f} | {gm:>9.4f} | {gc:>9.4f}")
    m, c = m - lr * gm, c - lr * gc

### Watch it learn, step by step
Press **▶ play**, drag **step**, or click **next step**. Try a bigger learning rate $\eta$ (0.30 and above) to see it overshoot.

In [ ]:
MAX_STEPS = 100

def scene_9_descent():
    Mg, Cg = np.meshgrid(np.linspace(-3, 5, 200), np.linspace(-5, 7, 200))
    Lg = np.mean((Mg[..., None] * x_data + Cg[..., None] - y_data) ** 2, axis=-1)
    xl = np.array([-3.3, 3.3])

    fig = window("9. Gradient descent", (18, 9.5))
    a1, a2, a3 = panels(fig, 4, ncols=3, top=0.88)

    a1.scatter(x_data, y_data, color=BLUE, zorder=3)
    residuals = a1.vlines(x_data, y_data, y_data, color=RED, alpha=0.45, lw=1.2)
    fit, = a1.plot(xl, xl, color=ORANGE)
    a1.set_xlim(-3.5, 3.5); a1.set_ylim(-9, 11); a1.set_xlabel("x"); a1.set_ylabel("y")

    a2.contour(Mg, Cg, Lg, levels=np.geomspace(0.55, 150, 16), cmap="viridis", linewidths=1.2)
    a2.plot(m_best, c_best, "*", color=GREEN, ms=18, label="best possible line")
    trail, = a2.plot([], [], ".-", color=ORANGE, ms=6, lw=1.5)
    here, = a2.plot([], [], "o", color=RED, ms=10, zorder=5)
    a2.set_xlim(-3, 5); a2.set_ylim(-5, 7); a2.set_xlabel("m"); a2.set_ylabel("c"); a2.legend(loc="upper left")

    curve, = a3.plot([], [], color=PURPLE)
    now, = a3.plot([], [], "o", color=RED, ms=9)
    a3.set_yscale("log"); a3.set_xlim(0, MAX_STEPS); a3.set_xlabel("step")

    step = slider(fig, 3, "step", 0, MAX_STEPS, 0, step=1, fmt="%d")
    lr = slider(fig, 2, "learning rate η", 0.01, 0.35, 0.05, step=0.01)
    m_start = slider(fig, 1, "start m", -3, 5, -1.5, step=0.1, fmt="%.1f")
    c_start = slider(fig, 0, "start c", -5, 7, -3.0, step=0.1, fmt="%.1f")
    play_button(fig, 3, step, MAX_STEPS, ms=150)
    button(fig, 3, "next step", lambda: step.set_val(min(step.val + 1, MAX_STEPS)), col=1)
    button(fig, 2, "back to 0", lambda: step.set_val(0))
    arrow = {"a": None}

    def draw():
        k = int(step.val)
        path = gradient_descent(m_start.val, c_start.val, lr.val, MAX_STEPS)
        losses = np.array([loss(*p) for p in path])
        m, c = path[k]
        gm, gc = gradient(m, c)

        fit.set_ydata(m * xl + c)
        fit.set_label(line_eq(m, c)); a1.legend(loc="upper left")
        residuals.set_segments([[(xi, yi), (xi, m * xi + c)] for xi, yi in zip(x_data, y_data)])
        a1.set_title(f"step {k}: the line")

        trail.set_data(path[:k + 1, 0], path[:k + 1, 1])
        here.set_data([m], [c])
        if arrow["a"] is not None:
            arrow["a"].remove()
            arrow["a"] = None
        if k < MAX_STEPS:
            nm, nc = path[k + 1]
            arrow["a"] = a2.annotate("", xy=(nm, nc), xytext=(m, c),
                                     arrowprops=dict(arrowstyle="->", color=RED, lw=2.5))
        nz = lambda v: 0.0 if abs(v) < 5e-4 else v
        a2.set_title(f"loss landscape L(m, c)\n∂L/∂m = {nz(gm):.4g},  ∂L/∂c = {nz(gc):.4g}")

        curve.set_data(np.arange(k + 1), losses[:k + 1])
        now.set_data([k], [losses[k]])
        a3.set_ylim(0.3, max(100, losses[:k + 1].max() * 2))
        a3.set_title(f"loss = {losses[k]:.4g}")
        fig.canvas.draw_idle()

    live(draw, step, lr, m_start, c_start)
    finish(fig)

scene_9_descent()

In [ ]:
# The same journey on the 3D loss surface: a ball rolling down into the valley
path = gradient_descent(m0, c0, 0.05, MAX_STEPS)
Ms, Cs = np.meshgrid(np.linspace(-3, 5, 60), np.linspace(-5, 7, 60))
Ls = np.mean((Ms[..., None] * x_data + Cs[..., None] - y_data) ** 2, axis=-1)

fig = window("9. The loss surface", (13, 9.5))
ax = axes3d(fig)
ax.plot_surface(Ms, Cs, Ls, cmap="viridis", alpha=0.75, linewidth=0)
ax.plot(path[:, 0], path[:, 1], [loss(*p) + 0.5 for p in path], ".-", color=ORANGE, ms=5, lw=3, zorder=10)
ax.plot([m_best], [c_best], [loss(m_best, c_best)], "*", color=GREEN, ms=16, zorder=11)
ax.set_xlabel("m"); ax.set_ylabel("c"); ax.set_zlabel("loss")
ax.set_title("the loss surface L(m, c) and the path of gradient descent")
ax.view_init(elev=30, azim=-50)
spin_button(fig, ax)
finish(fig)

---
# 10. The chain rule with more than one variable

$$z = f(x, y), \qquad x = x(t), \qquad y = y(t)$$

A nudge in $t$ moves **both** $x$ and $y$, and each of them moves $z$. Add up the two paths:

$$\frac{dz}{dt} = \frac{\partial f}{\partial x}\,\frac{dx}{dt} + \frac{\partial f}{\partial y}\,\frac{dy}{dt}$$

Example: walk around the unit circle $x = \cos t,\; y = \sin t$ on the surface $f(x, y) = x^2 y^3$.

In [ ]:
def x_of_t(t): return np.cos(t)
def y_of_t(t): return np.sin(t)
def z_of_t(t): return F(x_of_t(t), y_of_t(t))

def chain_parts(t):
    x, y = x_of_t(t), y_of_t(t)
    fx, fy = 2 * x * y ** 3, 3 * x ** 2 * y ** 2       # partial derivatives of f
    dx, dy = -np.sin(t), np.cos(t)                    # how x and y move with t
    return fx, fy, dx, dy

Xc, Yc = np.meshgrid(np.linspace(-1.4, 1.4, 50), np.linspace(-1.4, 1.4, 50))
Zc = F(Xc, Yc)
tt = np.linspace(0, 2 * np.pi, 400)

fig = window("10. A walk on the surface", (13, 9.5))
ax = axes3d(fig)
ax.plot_surface(Xc, Yc, Zc, cmap="RdBu_r", norm=TwoSlopeNorm(vmin=Zc.min(), vcenter=0, vmax=Zc.max()),
                alpha=0.5, linewidth=0)
ax.plot(x_of_t(tt), y_of_t(tt), z_of_t(tt), color="k", lw=3.5, zorder=10, label="z(t): the path on the surface")
ax.plot(x_of_t(tt), y_of_t(tt), np.full_like(tt, Zc.min()), "--", color=GRAY, lw=2, zorder=9,
        label="(x(t), y(t)): the walk on the floor")
ax.set_title("walking around a circle on f(x, y) = x² y³"); ax.legend(loc="upper left")
label3d(ax)
spin_button(fig, ax)
finish(fig)

In [ ]:
def scene_10_walk():
    fig = window("10. Multivariable chain rule", (17, 8.5))
    a1, a2 = panels(fig, 1, ncols=2, top=0.86)
    a1.contourf(Xc, Yc, Zc, levels=30, cmap="RdBu_r", alpha=0.75)
    a1.plot(x_of_t(tt), y_of_t(tt), color="k", lw=1.5)
    spot, = a1.plot([], [], "o", color="k", ms=9)
    q_walk = a1.quiver([0], [0], [0], [0], color=ORANGE, angles="xy", scale_units="xy", scale=1, width=0.008)
    q_grad = a1.quiver([0], [0], [0], [0], color=GREEN, angles="xy", scale_units="xy", scale=1, width=0.008)
    a1.set_aspect("equal"); a1.set_xlim(-1.4, 1.4); a1.set_ylim(-1.4, 1.4); a1.grid(False)
    a1.set_title("the walk seen from above")

    a2.plot(tt, z_of_t(tt), color=BLUE, label="z(t) = f(cos t, sin t)")
    tangent, = a2.plot([], [], "--", color=PURPLE, lw=1.8)
    zdot, = a2.plot([], [], "o", color=RED, ms=9)
    a2.set_ylim(-0.6, 0.6); a2.set_xlabel("t"); a2.legend(loc="upper right")

    t_s = slider(fig, 0, "t0", 0, 6.28, 1.0, step=0.02)
    play_button(fig, 0, t_s, 6.28, ms=60, loop=True, label="▶ walk")

    def draw():
        t = t_s.val
        fx, fy, dx, dy = chain_parts(t)
        rate = fx * dx + fy * dy
        px, py = x_of_t(t), y_of_t(t)
        spot.set_data([px], [py])
        q_walk.set_offsets([[px, py]]); q_walk.set_UVC([0.5 * dx], [0.5 * dy])
        q_grad.set_offsets([[px, py]]); q_grad.set_UVC([0.25 * fx], [0.25 * fy])
        a1.legend(handles=[plt.Line2D([], [], color=ORANGE, lw=3, label=f"direction of walking (dx/dt, dy/dt) = ({dx:.2f}, {dy:.2f})"),
                           plt.Line2D([], [], color=GREEN, lw=3, label=f"gradient (∂f/∂x, ∂f/∂y) = ({fx:.2f}, {fy:.2f})")],
                  loc="lower left", fontsize=10)
        tangent.set_data(tt, z_of_t(t) + rate * (tt - t))
        zdot.set_data([t], [z_of_t(t)])
        h = 1e-6
        a2.set_title(f"∂f/∂x·dx/dt + ∂f/∂y·dy/dt = {fx * dx:.4f} + {fy * dy:.4f} = {rate:.4f}\n"
                     f"measured Δz/Δt = {(z_of_t(t + h) - z_of_t(t)) / h:.4f}", fontsize=13)
        fig.canvas.draw_idle()

    live(draw, t_s)
    finish(fig)

scene_10_walk()

In [ ]:
t0 = 1.0
fx, fy, dx, dy = chain_parts(t0)
print(f"path through x :  ∂f/∂x · dx/dt = {fx:.5f} × {dx:.5f} = {fx * dx:.6f}")
print(f"path through y :  ∂f/∂y · dy/dt = {fy:.5f} × {dy:.5f} = {fy * dy:.6f}")
print(f"sum            :  {fx * dx + fy * dy:.6f}\n")
slope_table(z_of_t, t0, exact=fx * dx + fy * dy, name="t", out="z")

### Back to the wind turbine from part 5

$g = u^2\,t + 5$ with $u = 2t^2$. It is $z = f(x, y)$ with $x = u(t)$ and $y = t$:

$$\frac{dg}{dt} = \underbrace{\frac{\partial g}{\partial u}\frac{du}{dt}}_{\text{path through } u} + \underbrace{\frac{\partial g}{\partial t}\frac{dt}{dt}}_{\text{direct path}} = (2ut)(4t) + u^2\cdot 1$$

In [ ]:
t0 = 1.0
u0 = wind(t0)
path_u = (2 * u0 * t0) * (4 * t0)
path_t = u0 ** 2 * 1
print(f"path through u = {path_u},  direct path = {path_t},  total = {path_u + path_t}")
print(f"measured       = {(g_two_paths(t0 + 1e-6) - g_two_paths(t0)) / 1e-6:.4f}")

---
# 11. Backpropagation = the chain rule, applied backwards

### 11.1 One neuron, one data point

$$z = w\,x + b \qquad a = \sigma(z) = \frac{1}{1 + e^{-z}} \qquad L = (a - y)^2$$

To learn we need $\dfrac{\partial L}{\partial w}$ and $\dfrac{\partial L}{\partial b}$. $w$ reaches $L$ through a chain, so:

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial a}\cdot\frac{\partial a}{\partial z}\cdot\frac{\partial z}{\partial w} = 2(a - y)\cdot a(1 - a)\cdot x$$

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x_in, y_true = 1.5, 1.0      # one training example
w, b = 0.8, -0.5             # current weights

# ---- forward pass: left to right ----
z = w * x_in + b
a = sigmoid(z)
L = (a - y_true) ** 2

# ---- backward pass: right to left, multiplying local slopes ----
dL_da = 2 * (a - y_true)
da_dz = a * (1 - a)          # slope of the sigmoid
dL_dz = dL_da * da_dz
dL_dw = dL_dz * x_in         # dz/dw = x
dL_db = dL_dz * 1            # dz/db = 1

print(f"forward : z = {z:.4f}   a = {a:.4f}   L = {L:.4f}")
print(f"backward: dL/da = {dL_da:.4f}   da/dz = {da_dz:.4f}   dL/dz = {dL_dz:.4f}")
print(f"          dL/dw = {dL_dw:.5f}   dL/db = {dL_db:.5f}")

# check with the "tiny nudge" method
h = 1e-6
Lw = (sigmoid((w + h) * x_in + b) - y_true) ** 2
Lb = (sigmoid(w * x_in + b + h) - y_true) ** 2
print(f"\nnudge w: {(Lw - L) / h:.5f}      nudge b: {(Lb - L) / h:.5f}")

In [ ]:
fig = window("11. Forward and backward pass", (15, 6.5))
ax = fig.add_axes([0.02, 0.02, 0.96, 0.96])
ax.set_xlim(0, 15); ax.set_ylim(0, 6); ax.axis("off")
nodes = {"w": (1, 5), "x": (1, 3), "b": (1, 1), "z": (5.2, 3), "a": (9.2, 3), "L": (13.2, 3)}
text = {"w": f"w = {w}", "x": f"x = {x_in}", "b": f"b = {b}",
        "z": "z = w·x + b", "a": "a = σ(z)", "L": "L = (a − y)²"}
fwd = {"z": f"{z:.3f}", "a": f"{a:.3f}", "L": f"{L:.4f}"}
bwd = {"w": f"∂L/∂w = {dL_dw:.4f}", "b": f"∂L/∂b = {dL_db:.4f}", "z": f"∂L/∂z = {dL_dz:.4f}",
       "a": f"∂L/∂a = {dL_da:.4f}", "L": "∂L/∂L = 1"}
for src, dst in [("w", "z"), ("x", "z"), ("b", "z"), ("z", "a"), ("a", "L")]:
    ax.annotate("", xy=(nodes[dst][0] - 1.1, nodes[dst][1]), xytext=(nodes[src][0] + 0.9, nodes[src][1]),
                arrowprops=dict(arrowstyle="->", lw=2, color=GRAY))
for k, (px, py) in nodes.items():
    ax.text(px, py, text[k], ha="center", va="center", fontsize=14,
            bbox=dict(boxstyle="round,pad=0.5", fc="white", ec=BLUE, lw=2))
    if k in fwd:
        ax.text(px, py + 0.75, fwd[k], ha="center", color=BLUE, fontsize=14, fontweight="bold")
    if k in bwd:
        ax.text(px, py - 0.8, bwd[k], ha="center", color=RED, fontsize=13, fontweight="bold")
ax.text(7.5, 5.6, "forward pass: values (blue), left → right", color=BLUE, ha="center", fontsize=14)
ax.text(7.5, 0.2, "backward pass: gradients (red), right → left, multiply the local slopes", color=RED, ha="center", fontsize=14)
finish(fig)

### 11.2 A real use case: "Is this temperature comfortable?"

Inputs: temperatures from 10 °C to 40 °C. Label: **1** if between 20 °C and 28 °C, else **0**.
A straight line can't draw a bump like that, so we use a tiny network with 2 hidden neurons:

$$h_1 = \sigma(w_1 x + b_1),\quad h_2 = \sigma(w_2 x + b_2),\qquad \hat y = \sigma(v_1 h_1 + v_2 h_2 + c),\qquad L = \frac{1}{n}\sum_i(\hat y_i - y_i)^2$$

Where does the **multivariable** chain rule show up?

| chain rule | in the network |
|---|---|
| $t$ | one weight, for example $w_1$ |
| $x(t),\ y(t)$ | the things it changes: the predictions $\hat y_1, \hat y_2, \dots$ on every data point |
| $z = f(x, y)$ | the loss, which depends on all predictions |
| $\dfrac{dz}{dt} = \dfrac{\partial f}{\partial x}\dfrac{dx}{dt} + \dfrac{\partial f}{\partial y}\dfrac{dy}{dt}$ | $\dfrac{\partial L}{\partial w_1} = \sum_i \dfrac{\partial L}{\partial \hat y_i}\dfrac{\partial \hat y_i}{\partial w_1}$ &nbsp; (add up every path) |

In [ ]:
temps = np.linspace(10, 40, 31)                                  # °C
comfy = ((temps >= 20) & (temps <= 28)).astype(float)            # 1 = comfortable
x_net = (temps - 25) / 7.5                                       # scale inputs to about [-2, 2]

def to_x(T):
    return (T - 25) / 7.5

def forward(p, x):
    z1 = np.outer(x, p["w"]) + p["b"]        # (n, 2) hidden neurons before σ
    h = sigmoid(z1)                          # (n, 2) hidden neurons
    z2 = h @ p["v"] + p["c"]                 # (n,)   output before σ
    y_hat = sigmoid(z2)                      # (n,)   prediction
    return h, y_hat

def backward(p, x, y, h, y_hat):
    n = len(x)
    dL_dyhat = 2 * (y_hat - y) / n                   # L = mean((ŷ - y)²)
    dL_dz2 = dL_dyhat * y_hat * (1 - y_hat)          # through the output σ
    grads = {"v": h.T @ dL_dz2,                      # z2 = v1·h1 + v2·h2 + c
             "c": dL_dz2.sum()}                      # .sum() = add up the paths through every data point
    dL_dh = np.outer(dL_dz2, p["v"])                 # each hidden neuron gets its share of the blame
    dL_dz1 = dL_dh * h * (1 - h)                     # through the hidden σ
    grads["w"] = (dL_dz1 * x[:, None]).sum(axis=0)   # z1 = w·x + b
    grads["b"] = dL_dz1.sum(axis=0)
    return grads

rng = np.random.default_rng(1)
params = {"w": rng.normal(0, 1, 2), "b": rng.normal(0, 1, 2), "v": rng.normal(0, 1, 2), "c": 0.0}
LR, EPOCHS, SNAP = 3.0, 3000, 20
history = []
for epoch in range(EPOCHS + 1):
    h, y_hat = forward(params, x_net)
    L = np.mean((y_hat - comfy) ** 2)
    if epoch % SNAP == 0:
        history.append((epoch, L, {k: np.copy(v) for k, v in params.items()}))
    grads = backward(params, x_net, comfy, h, y_hat)
    for k in params:
        params[k] = params[k] - LR * grads[k]            # gradient descent, exactly like part 9

print(f"final loss = {L:.5f}")
print(f"accuracy   = {np.mean((y_hat > 0.5) == comfy) * 100:.0f}%")

In [ ]:
def scene_11_training():
    T_fine = np.linspace(8, 42, 400)
    epochs = np.array([hh[0] for hh in history])
    all_losses = np.array([hh[1] for hh in history])

    fig = window("11. Training the network", (18, 8.5))
    a1, a2, a3 = panels(fig, 1, ncols=3, top=0.9)
    a1.scatter(temps, comfy, color=BLUE, zorder=3, label="data (1 = comfortable)")
    pred, = a1.plot([], [], color=ORANGE, label="network output ŷ")
    a1.axhline(0.5, color=GRAY, ls=":", lw=1)
    a1.set_xlim(8, 42); a1.set_ylim(-0.1, 1.25); a1.set_xlabel("temperature (°C)")
    a1.legend(loc="upper left", fontsize=10, ncol=2)
    hid1, = a2.plot([], [], color=GREEN, label="hidden neuron h1")
    hid2, = a2.plot([], [], color=PURPLE, label="hidden neuron h2")
    a2.set_xlim(8, 42); a2.set_ylim(-0.05, 1.05); a2.set_xlabel("temperature (°C)"); a2.legend(loc="center right", fontsize=10)
    a2.set_title("what each hidden neuron has learned")
    a3.plot(epochs, all_losses, color=GRAY, lw=1.2, alpha=0.5)
    progress, = a3.plot([], [], color=PURPLE)
    dot, = a3.plot([], [], "o", color=RED, ms=9)
    a3.set_yscale("log"); a3.set_xlabel("epoch")

    snap = slider(fig, 0, "snapshot", 0, len(history) - 1, 0, step=1, fmt="%d")
    play_button(fig, 0, snap, len(history) - 1, ms=60)

    def draw():
        i = int(snap.val)
        epoch, L, p = history[i]
        h, y_hat = forward(p, to_x(T_fine))
        pred.set_data(T_fine, y_hat)
        a1.set_title(f"epoch {epoch}: prediction")
        hid1.set_data(T_fine, h[:, 0]); hid2.set_data(T_fine, h[:, 1])
        progress.set_data(epochs[:i + 1], all_losses[:i + 1]); dot.set_data([epoch], [L])
        a3.set_title(f"loss = {L:.4f}")
        fig.canvas.draw_idle()

    live(draw, snap)
    finish(fig)

scene_11_training()

### 11.3 Trust, but verify: backprop vs. the tiny-nudge method

In [ ]:
def total_loss(p):
    _, y_hat = forward(p, x_net)
    return np.mean((y_hat - comfy) ** 2)

test = {k: np.copy(v) for k, v in history[5][2].items()}      # weights from early in training
h_, yh_ = forward(test, x_net)
bp = backward(test, x_net, comfy, h_, yh_)

eps = 1e-6
print(f"{'weight':>8} | {'backprop':>12} | {'nudge':>12} | {'difference':>10}")
print("-" * 52)
for k in ["w", "b", "v", "c"]:
    for j in range(np.size(test[k])):
        bumped = {kk: np.copy(vv) for kk, vv in test.items()}
        if np.ndim(bumped[k]) == 0:
            bumped[k] = bumped[k] + eps
            g = bp[k]
        else:
            bumped[k][j] += eps
            g = bp[k][j]
        nudge = (total_loss(bumped) - total_loss(test)) / eps
        label = k if np.ndim(test[k]) == 0 else f"{k}{j + 1}"
        print(f"{label:>8} | {g:>12.7f} | {nudge:>12.7f} | {abs(g - nudge):>10.1e}")
print("\nThe nudge method needs one extra forward pass PER WEIGHT.")
print("Backprop gets all of them from one forward pass + one backward pass.")

In [ ]:
# The z = f(x, y), x = x(t), y = y(t) pattern, literally:
# the output ŷ depends on h1 and h2, and BOTH depend on the input temperature T.
# We use the weights from epoch 200 (the hidden neurons are still soft, so both paths matter).
epoch200 = history[10][2]
T0 = 24.0
xx = to_x(T0)
h1, h2 = sigmoid(epoch200["w"] * xx + epoch200["b"])
yh = sigmoid(epoch200["v"][0] * h1 + epoch200["v"][1] * h2 + epoch200["c"])
dyh_dh1 = yh * (1 - yh) * epoch200["v"][0]          # ∂ŷ/∂h1
dyh_dh2 = yh * (1 - yh) * epoch200["v"][1]          # ∂ŷ/∂h2
dh1_dT = h1 * (1 - h1) * epoch200["w"][0] / 7.5     # dh1/dT
dh2_dT = h2 * (1 - h2) * epoch200["w"][1] / 7.5     # dh2/dT
eps = 1e-6
measured = (forward(epoch200, np.array([to_x(T0 + eps)]))[1][0] - forward(epoch200, np.array([xx]))[1][0]) / eps
print(f"path through h1 : ∂ŷ/∂h1 · dh1/dT = {dyh_dh1 * dh1_dT:+.5f}   (pushes 'comfortable' up)")
print(f"path through h2 : ∂ŷ/∂h2 · dh2/dT = {dyh_dh2 * dh2_dT:+.5f}   (pushes it down)")
print(f"sum             : {dyh_dh1 * dh1_dT + dyh_dh2 * dh2_dT:+.5f}")
print(f"measured dŷ/dT  : {measured:+.5f}   at T = {T0} °C, epoch 200")

In [ ]:
# Optional: PyTorch computes the same gradients automatically (skip if torch is not installed)
try:
    import torch
    tp = {k: torch.tensor(np.atleast_1d(v), dtype=torch.float64, requires_grad=True) for k, v in test.items()}
    xt = torch.tensor(x_net)[:, None]; yt = torch.tensor(comfy)
    ht = torch.sigmoid(xt * tp["w"] + tp["b"])
    yhat_t = torch.sigmoid(ht @ tp["v"] + tp["c"])
    ((yhat_t - yt) ** 2).mean().backward()
    for k in ["w", "b", "v", "c"]:
        print(f"{k}:  ours {np.round(np.atleast_1d(bp[k]), 7)}   torch {np.round(tp[k].grad.numpy(), 7)}")
except ImportError:
    print("PyTorch not installed, skipping (pip install torch to try it).")

---
# Recap

| idea | formula |
|---|---|
| slope of a line | $\dfrac{\Delta f}{\Delta x}$ |
| derivative | $\dfrac{df}{dx} = \lim\limits_{\Delta x \to 0}\dfrac{f(x+\Delta x) - f(x)}{\Delta x}$ |
| velocity | $v = \dfrac{ds}{dt}$ |
| chain rule | $\dfrac{dg}{dt} = \dfrac{dg}{du}\dfrac{du}{dt}$ |
| partial derivative | nudge one input, freeze the others |
| total derivative | $df = \dfrac{\partial f}{\partial x}dx + \dfrac{\partial f}{\partial y}dy$ |
| gradient descent | $\theta \leftarrow \theta - \eta\,\nabla L$ |
| multivariable chain rule | $\dfrac{dz}{dt} = \dfrac{\partial f}{\partial x}\dfrac{dx}{dt} + \dfrac{\partial f}{\partial y}\dfrac{dy}{dt}$ |
| backpropagation | the chain rule, applied from the loss backwards, adding up every path |

**So what is $dy/dx$?** Nudge the input a tiny bit, see how much the output moves, and divide.

In [ ]:
plt.close("all")   # close every plot window